# 03 — Which signals require an interface?

A five-arm factorial separates pointwise endpoint supervision from native-space
(H_0) supervision. The main output is the analysis table; Figure 4 visualizes
the remaining (H_0) residual and Figure A1 provides a qualitative MST view.

In [ ]:
# 1. Settings
from pathlib import Path

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO, INSTALL_REQUIREMENTS = True, True
PAIR = "qwen3_4b_to_bert_base"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"analysis_topology_{PAIR}_v2"
SEEDS = [42, 43, 44]
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
LAMBDA_H0 = 0.75
PROBE_EVERY, PROBE_SIZE = 250, 1024
MST_ROWS = 96
RESIDUAL_BATCH_SIZE, RESIDUAL_BATCHES = 64, 16
EXECUTE, STOP_ON_ERROR, REQUIRE_COMPLETE, RENDER_FIGURES = False, True, True, True
CUDA_VISIBLE_DEVICES = "0"

In [ ]:
# 2. Repo, dependencies, GPU và dữ liệu
import subprocess, sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
if EXECUTE:
    import torch
    assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi train."
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Training data: {TRAIN_DATA}")

In [ ]:
# 3. Plan: endpoint × H0 factorial plus the H0-source control
import shlex
from _analysis_common import PAIRS, collect_jobs, geoode_command, read_jsonl, run_jobs

pair, cache_dir = PAIRS[PAIR], PROJECT_DIR / "runs" / "teacher_cache"
run_root = PROJECT_DIR / "runs" / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
arms = {
    "no_teacher": (0, 0, "—"),
    "endpoint_only": (1, 0, "—"),
    "h0_only": (0, LAMBDA_H0, "Original"),
    "combined_projected": (1, LAMBDA_H0, "PCA"),
    "combined_original": (1, LAMBDA_H0, "Original"),
}
jobs = []
for arm, (lambda_end, lambda_h0, source) in arms.items():
    for seed in SEEDS:
        run_dir = run_root / arm / f"seed_{seed}"
        uses_endpoint = bool(lambda_end)
        gauge = (["--gauge_align", "--gauge_rotation", "procrustes", "--gauge_refit_every", 1]
                 if uses_endpoint else ["--no-gauge_align", "--gauge_refit_every", 0])
        extra = ["--projection_type", "pca", *gauge, "--lambda_end", lambda_end,
                 "--lambda_ctr", 0, "--lambda_topo", lambda_h0, "--lambda_h1", 0,
                 "--topo_metric", "chord", "--topo_teacher_source",
                 "projected" if source == "PCA" else "original",
                 "--probe_every", PROBE_EVERY, "--probe_size", PROBE_SIZE,
                 "--no_eval_retrieval"]
        jobs.append({
            "name": f"{arm}/seed_{seed}", "arm": arm, "seed": seed,
            "lambda_end": lambda_end, "lambda_h0": lambda_h0, "h0_source": source,
            "run_dir": run_dir,
            "command": geoode_command(
                PROJECT_DIR, pair=pair, train_data=TRAIN_DATA, cache_dir=cache_dir,
                run_dir=run_dir, seed=seed, batch_size=BATCH_SIZE, epochs=EPOCHS,
                learning_rate=LR, extra=extra,
            ),
        })
print(f"Plan: {len(jobs)} jobs -> {run_root}")
for job in jobs:
    print(shlex.join(job["command"]))

In [ ]:
# 4. Chạy tuần tự; final-test record là resume boundary
from IPython.display import display

if EXECUTE:
    display(run_jobs(
        PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
        stop_on_error=STOP_ON_ERROR,
    ))
else:
    print("Dry run: đặt EXECUTE=True để chạy các job còn thiếu.")

In [ ]:
# 5. Main analysis table — endpoint and intrinsic topology
import numpy as np
import pandas as pd

results = collect_jobs(jobs)
probes = []
for job in jobs:
    history = read_jsonl(Path(job["run_dir"]) / "probe_metrics.jsonl")
    if not history.empty:
        final = history.sort_values("global_step").iloc[-1]
        probes.append({
            "arm": job["arm"], "seed": job["seed"],
            "endpoint_error": 1 - final.get("probe_cos_target", np.nan),
            "h0_residual": final.get("probe_h0_w1_teacher", np.nan),
            "probe_step": final.get("global_step", np.nan),
        })
by_run = results.merge(pd.DataFrame(probes), on=["arm", "seed"], how="left")
by_run.to_csv(run_root / "decomposition_by_run.csv", index=False)
done = by_run.query("status == 'done'").copy()
if done.empty:
    print("No completed runs yet.")
else:
    counts = done.groupby("arm").size().to_dict()
    expected = {arm: len(SEEDS) for arm in arms}
    if REQUIRE_COMPLETE and counts != expected:
        raise RuntimeError(f"Incomplete decomposition grid: got {counts}, expected {expected}")
    if REQUIRE_COMPLETE and len(probes) != len(jobs):
        raise RuntimeError(f"Missing probe records: got {len(probes)}, expected {len(jobs)}")
    stats = done.groupby("arm").agg(
        endpoint_mean=("endpoint_error", "mean"), endpoint_sd=("endpoint_error", "std"),
        h0_mean=("h0_residual", "mean"), h0_sd=("h0_residual", "std"),
        avg_mean=("avg_all", "mean"), avg_sd=("avg_all", "std"), n=("avg_all", "count"),
    )
    order = ["no_teacher", "endpoint_only", "h0_only", "combined_projected", "combined_original"]
    rows = []
    for arm in order:
        values = stats.loc[arm]
        lambda_end, lambda_h0, source = arms[arm]
        rows.append({
            r"$L_{end}$": "✓" if lambda_end else "—",
            r"$L_{H_0}$": "✓" if lambda_h0 else "—",
            r"$H_0$ source": source,
            "Endpoint error ↓": "—" if not lambda_end else f"{values.endpoint_mean:.4f} ± {values.endpoint_sd:.4f}",
            r"$H_0$ residual ↓": f"{values.h0_mean:.4f} ± {values.h0_sd:.4f}",
            "AVG ↑": f"{100 * values.avg_mean:.2f} ± {100 * values.avg_sd:.2f}",
            "n": int(values.n),
        })
    table = pd.DataFrame(rows)
    table.to_csv(run_root / "table_2_signal_decomposition.csv", index=False)
    (run_root / "table_2_signal_decomposition.tex").write_text(
        table.drop(columns="n").to_latex(index=False, escape=False), encoding="utf-8"
    )
    display(table)

In [ ]:
# 6. Figure 4 and Appendix Figure A1 (render only after completed runs)
if RENDER_FIGURES and not done.empty:
    import gc
    import matplotlib.pyplot as plt
    import torch
    from scipy.sparse.csgraph import minimum_spanning_tree
    from transformers import AutoTokenizer
    from _analysis_common import final_checkpoint, load_teacher_cache, set_paper_style, teacher_cache_path
    from src import structural_audit as audit
    from src.criterions.h0_topological_loss import h0_death_times, pairwise_distance

    set_paper_style()
    frame = pd.read_csv(TRAIN_DATA)
    text_col = "text" if "text" in frame else "premise"
    n_rows = min(len(frame), RESIDUAL_BATCH_SIZE * RESIDUAL_BATCHES)
    n_rows = (n_rows // RESIDUAL_BATCH_SIZE) * RESIDUAL_BATCH_SIZE
    rng = np.random.default_rng(0)
    indices = np.sort(rng.choice(len(frame), n_rows, replace=False))
    texts = frame.iloc[indices][text_col].astype(str).tolist()
    teacher, _ = load_teacher_cache(teacher_cache_path(PROJECT_DIR, cache_dir, pair=pair, train_data=TRAIN_DATA))
    teacher = teacher[torch.as_tensor(indices)].float()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(pair["student"])
    clouds = {"Teacher": teacher}
    for label, arm in (("Endpoint only", "endpoint_only"), (r"Endpoint + $H_0$", "combined_original")):
        model = audit.load_student(pair["student"], final_checkpoint(run_root / arm / f"seed_{SEEDS[0]}", EPOCHS), device=device)
        clouds[label] = audit.encode_texts(model, tokenizer, texts, device=device,
                                           pooling=pair["student_pooling"], batch_size=64)["final"].float()
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    base = teacher[:MST_ROWS]
    centered = audit.unit(base) - audit.unit(base).mean(0)
    _, _, vh = torch.linalg.svd(centered, full_matrices=False)
    xy = (centered @ vh[:2].T).numpy()
    def mst_edges(cloud):
        distance = pairwise_distance(audit.unit(cloud[:MST_ROWS]), metric="chord").numpy() + 1
        np.fill_diagonal(distance, 0)
        tree = minimum_spanning_tree(distance).tocoo()
        return zip(tree.row, tree.col)
    fig, axes = plt.subplots(1, 3, figsize=(5.5, 1.85))
    for ax, ((name, cloud), color) in zip(axes, zip(clouds.items(), ["#6B46C1", "#DD6B20", "#2F855A"])):
        ax.plot(xy[:, 0], xy[:, 1], "o", ms=2.2, color="#1F2937", markeredgewidth=0, zorder=2)
        for i, j in mst_edges(cloud):
            ax.plot(xy[[i, j], 0], xy[[i, j], 1], color=color, lw=.7, alpha=.8)
        ax.set_title(name)
        ax.set_axis_off()
    fig.text(.5, .015, "Fixed layout; MST edges are recomputed in each native space.",
             ha="center", fontsize=6.5, color="#6B7280")
    fig.tight_layout(rect=(0, .07, 1, 1))
    fig.savefig(run_root / "figure_A1_h0_mst.pdf", bbox_inches="tight")
    fig.savefig(run_root / "figure_A1_h0_mst.png", dpi=300, bbox_inches="tight")
    plt.show()

    def residual_map(student):
        return np.stack([
            np.abs(
                h0_death_times(student[start:start + RESIDUAL_BATCH_SIZE], metric="chord").cpu().numpy()
                - h0_death_times(teacher[start:start + RESIDUAL_BATCH_SIZE], metric="chord").cpu().numpy()
            )
            for start in range(0, n_rows, RESIDUAL_BATCH_SIZE)
        ])
    residuals = {name: residual_map(cloud) for name, cloud in clouds.items() if name != "Teacher"}
    np.savez(run_root / "h0_absolute_residuals.npz", **{name.replace(" ", "_").replace("$", ""): value for name, value in residuals.items()})
    vmax = np.quantile(residuals["Endpoint only"], .99)
    fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.25), sharex=True, sharey=True)
    for ax, (name, values) in zip(axes, residuals.items()):
        image = ax.imshow(values, aspect="auto", cmap="Reds", vmin=0, vmax=vmax,
                          interpolation="nearest", rasterized=True)
        ax.set(title=name, xlabel=r"Sorted $H_0$ death rank")
        ax.text(.97, .95, f"median = {np.median(values):.3f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=7, color="#7F1D1D")
    axes[0].set_ylabel("Fixed evaluation mini-batch")
    fig.colorbar(image, ax=axes, label=r"$|\delta^S-\delta^T|$", fraction=.035, pad=.03)
    fig.subplots_adjust(left=.11, right=.90, bottom=.20, top=.86, wspace=.10)
    fig.savefig(run_root / "figure_4_h0_residual.pdf", bbox_inches="tight")
    fig.savefig(run_root / "figure_4_h0_residual.png", dpi=300, bbox_inches="tight")
    plt.show()